# Pipeline 1: Donor Churn / Lapse Risk Classifier

## 1. Problem Framing

### Business Problem
The organization depends entirely on donations to operate its safehouses. The founders have explicitly stated that they "lose donors and don't always understand why." Donor attrition without warning is one of the greatest operational risks because each lost donor directly reduces the organization's ability to house, feed, educate, and rehabilitate the girls it serves.

### Who Cares
- **Executive leadership / founders**: Need to understand how healthy their donor base is and which donors are at risk of leaving.
- **Staff managing donor outreach**: Need to know where to focus limited time — who should receive a personal thank-you call, a campaign invitation, or a re-engagement email?
- **The girls in the safehouses**: Every retained donor means continued funding for meals, education, counseling, and medical care.

### Why It Matters
With no dedicated marketing team and limited staff, the organization cannot afford to treat all donors the same. A data-driven early warning system that flags at-risk donors allows staff to intervene proactively with personalized outreach, rather than discovering months later that a supporter has silently stopped giving.

### Approach: Predictive AND Explanatory
- **Predictive goal**: Build a binary classifier that predicts whether an active monetary donor will lapse (no donation within 180 days of a reference date). This drives the operational alert system in the web app.
- **Explanatory goal**: Build an interpretable logistic regression to understand *which factors* most strongly drive donor retention. Coefficients from this model will inform the organization's donor strategy — e.g., "recurring donors are X% less likely to lapse" or "donors acquired through social media have higher churn risk."

Both models serve different purposes and are evaluated differently, following the textbook's distinction between prediction and explanation (Ch. 1, 9-11).

## 2. Data Acquisition, Preparation & Exploration

We will load the `supporters` and `donations` tables, filter to monetary donations, and engineer a donor-level summary with RFM-style features plus organizational context.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.figsize'] = (12, 6)

# ----- Load data -----
DATA_DIR = "../lighthouse_csv_v7"  # Adjust this path relative to your notebook location  # adjust path as needed for your repo
supporters = pd.read_csv(f"{DATA_DIR}/supporters.csv")
donations = pd.read_csv(f"{DATA_DIR}/donations.csv")

print(f"Supporters: {supporters.shape}")
print(f"Donations:  {donations.shape}")

In [ ]:
# Filter to monetary donations only
donations['donation_date'] = pd.to_datetime(donations['donation_date'])
donations['amount'] = pd.to_numeric(donations['amount'], errors='coerce')
monetary = donations[donations['donation_type'] == 'Monetary'].copy()

print(f"Monetary donations: {len(monetary)}")
print(f"Date range: {monetary['donation_date'].min()} to {monetary['donation_date'].max()}")
print(f"Unique donors: {monetary['supporter_id'].nunique()}")

### Feature Engineering

We build a donor-level summary table with RFM (Recency, Frequency, Monetary) features plus contextual features from the supporters table. We use a reference date approach: features are computed as of a cutoff date, and the label is whether the donor donated in the 180 days after that cutoff.

In [ ]:
# Define reference date — use a date that gives us enough history and enough future
REFERENCE_DATE = pd.Timestamp('2025-06-01')
LAPSE_WINDOW_DAYS = 180

# Split donations into history (before reference) and future (after reference)
hist = monetary[monetary['donation_date'] <= REFERENCE_DATE].copy()
future = monetary[(monetary['donation_date'] > REFERENCE_DATE) & 
                  (monetary['donation_date'] <= REFERENCE_DATE + pd.Timedelta(days=LAPSE_WINDOW_DAYS))].copy()

print(f"Historical donations (before {REFERENCE_DATE.date()}): {len(hist)}")
print(f"Future donations (next {LAPSE_WINDOW_DAYS} days): {len(future)}")
print(f"Donors with history: {hist['supporter_id'].nunique()}")

In [ ]:
# Build donor-level features from historical donations
donor_features = hist.groupby('supporter_id').agg(
    donation_count=('donation_id', 'count'),
    total_amount=('amount', 'sum'),
    avg_amount=('amount', 'mean'),
    max_amount=('amount', 'max'),
    min_amount=('amount', 'min'),
    std_amount=('amount', 'std'),
    first_donation=('donation_date', 'min'),
    last_donation=('donation_date', 'max'),
    has_recurring=('is_recurring', 'max'),
    campaign_count=('campaign_name', lambda x: x.notna().sum()),
    unique_channels=('channel_source', 'nunique'),
).reset_index()

# Recency: days since last donation as of reference date
donor_features['recency_days'] = (REFERENCE_DATE - donor_features['last_donation']).dt.days
# Tenure: days since first donation
donor_features['tenure_days'] = (REFERENCE_DATE - donor_features['first_donation']).dt.days
# Average days between donations
donor_features['avg_days_between'] = donor_features['tenure_days'] / donor_features['donation_count'].clip(lower=1)
# Fill std for single-donation donors
donor_features['std_amount'] = donor_features['std_amount'].fillna(0)

# Merge supporter-level context
donor_features = donor_features.merge(
    supporters[['supporter_id', 'supporter_type', 'relationship_type', 'acquisition_channel', 'status']],
    on='supporter_id', how='left'
)

# Create target: did the donor donate in the future window?
future_donors = set(future['supporter_id'].unique())
donor_features['lapsed'] = (~donor_features['supporter_id'].isin(future_donors)).astype(int)

print(f"Donor feature table: {donor_features.shape}")
print(f"\nTarget distribution:")
print(donor_features['lapsed'].value_counts())
print(f"\nLapse rate: {donor_features['lapsed'].mean():.1%}")

### Exploratory Analysis

In [ ]:
# Univariate statistics for all donor features
print('Donor Feature Summary Statistics:')
print('=' * 60)
print(donor_features[['recency_days', 'donation_count', 'total_amount', 'avg_amount',
                      'tenure_days', 'avg_days_between', 'std_amount', 'campaign_count']].describe().round(2))
print()
print('Missing values per column:')
print(donor_features.isnull().sum()[donor_features.isnull().sum() > 0])
if donor_features.isnull().sum().sum() == 0:
    print('No missing values found.')


In [ ]:
# Distribution of key features
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

axes[0,0].hist(donor_features['recency_days'], bins=15, color='#1f77b4', edgecolor='white')
axes[0,0].set_title('Recency (Days Since Last Gift)')

axes[0,1].hist(donor_features['donation_count'], bins=15, color='#2ca02c', edgecolor='white')
axes[0,1].set_title('Donation Frequency')

axes[0,2].hist(donor_features['total_amount'], bins=15, color='#ff7f0e', edgecolor='white')
axes[0,2].set_title('Total Monetary Value (PHP)')

sns.barplot(data=donor_features, x='lapsed', y='recency_days', ax=axes[1,0], palette='Set2')
axes[1,0].set_title('Recency by Lapse Status')
axes[1,0].set_xticklabels(['Retained', 'Lapsed'])

sns.barplot(data=donor_features, x='lapsed', y='donation_count', ax=axes[1,1], palette='Set2')
axes[1,1].set_title('Frequency by Lapse Status')
axes[1,1].set_xticklabels(['Retained', 'Lapsed'])

sns.barplot(data=donor_features, x='lapsed', y='has_recurring', ax=axes[1,2], palette='Set2')
axes[1,2].set_title('Recurring Rate by Lapse Status')
axes[1,2].set_xticklabels(['Retained', 'Lapsed'])

plt.tight_layout()
plt.show()

In [ ]:
# Lapse rate by acquisition channel
channel_lapse = donor_features.groupby('acquisition_channel').agg(
    donors=('supporter_id', 'count'),
    lapse_rate=('lapsed', 'mean'),
    avg_amount=('total_amount', 'mean')
).round(3).sort_values('lapse_rate', ascending=False)
print("Lapse rate by acquisition channel:")
print(channel_lapse)

print("\nLapse rate by recurring status:")
print(donor_features.groupby('has_recurring')['lapsed'].mean().round(3))

print("\nLapse rate by relationship type:")
print(donor_features.groupby('relationship_type')['lapsed'].mean().round(3))

In [ ]:
# Correlation matrix of numeric features
numeric_cols = ['recency_days', 'donation_count', 'total_amount', 'avg_amount', 
                'tenure_days', 'avg_days_between', 'has_recurring', 'campaign_count', 'lapsed']
corr = donor_features[numeric_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, square=True)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## 3. Modeling & Feature Selection

We build both an explanatory model (Logistic Regression with interpretable coefficients) and predictive models (Decision Tree, Random Forest, Gradient Boosted Trees). We compare their performance and use feature importance to understand what matters most.

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score, 
                             roc_curve, ConfusionMatrixDisplay, RocCurveDisplay)
import joblib

# Define features
numeric_features = ['recency_days', 'donation_count', 'total_amount', 'avg_amount',
                    'tenure_days', 'avg_days_between', 'has_recurring', 'campaign_count',
                    'std_amount', 'unique_channels']
categorical_features = ['acquisition_channel', 'relationship_type']

X = donor_features[numeric_features + categorical_features].copy()
y = donor_features['lapsed'].copy()

# Preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), categorical_features)
    ]
)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
print(f"Train: {X_train.shape[0]} samples, Test: {X_test.shape[0]} samples")
print(f"Train lapse rate: {y_train.mean():.1%}, Test lapse rate: {y_test.mean():.1%}")

### Explanatory Model: Logistic Regression

The purpose of this model is to understand which factors drive donor lapse. We care about the sign and magnitude of coefficients, not just predictive accuracy.

In [ ]:
# Explanatory logistic regression
log_pipe = Pipeline([
    ('prep', preprocessor),
    ('clf', LogisticRegression(random_state=42, max_iter=1000, C=1.0))
])

log_pipe.fit(X_train, y_train)

# Extract coefficients
feature_names = numeric_features + list(
    log_pipe.named_steps['prep'].transformers_[1][1].get_feature_names_out(categorical_features)
)
coefs = pd.DataFrame({
    'feature': feature_names,
    'coefficient': log_pipe.named_steps['clf'].coef_[0],
    'odds_ratio': np.exp(log_pipe.named_steps['clf'].coef_[0])
}).sort_values('coefficient', ascending=True)

print("Logistic Regression Coefficients (Explanatory Model)")
print("=" * 60)
print("Positive coefficient = increases lapse probability")
print("Negative coefficient = decreases lapse probability (protective)")
print()
print(coefs.to_string(index=False))

### VIF Check for Multicollinearity

For the explanatory model, we check Variance Inflation Factors (VIF) to ensure our coefficient estimates are not distorted by multicollinearity. Features with VIF > 10 are candidates for removal from the causal model, though they may still be useful in predictive models.


In [ ]:
# VIF check for multicollinearity in explanatory model
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm

# Get the processed numeric features for VIF
X_train_processed = log_pipe.named_steps['prep'].transform(X_train)
X_vif = pd.DataFrame(X_train_processed, columns=feature_names)
X_vif = sm.add_constant(X_vif)

vif_data = pd.DataFrame({
    'Feature': X_vif.columns[1:],  # skip constant
    'VIF': [variance_inflation_factor(X_vif.values, i+1) for i in range(len(X_vif.columns)-1)]
}).sort_values('VIF', ascending=False)

print('Variance Inflation Factors (VIF > 10 indicates problematic multicollinearity):')
print('=' * 50)
print(vif_data.to_string(index=False))

high_vif = vif_data[vif_data['VIF'] > 10]
if len(high_vif) > 0:
    print(f'\nWARNING: {len(high_vif)} features with VIF > 10. Consider removing for causal interpretation.')
    print('High VIF features:', high_vif['Feature'].tolist())
else:
    print('\nNo features with VIF > 10. Multicollinearity is not a major concern.')


In [ ]:
# Visualize coefficients
fig, ax = plt.subplots(figsize=(10, 7))
colors = ['#2ca02c' if c < 0 else '#d62728' for c in coefs['coefficient']]
ax.barh(coefs['feature'], coefs['coefficient'], color=colors)
ax.set_xlabel('Coefficient (log-odds)')
ax.set_title('Logistic Regression Coefficients: Drivers of Donor Lapse')
ax.axvline(x=0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

### Predictive Models: Decision Tree, Random Forest, Gradient Boosting

These models prioritize out-of-sample prediction accuracy for the operational alert system.

In [ ]:
# Build multiple predictive models
models = {
    'Logistic Regression': Pipeline([('prep', preprocessor), 
        ('clf', LogisticRegression(random_state=42, max_iter=1000))]),
    'Decision Tree': Pipeline([('prep', preprocessor), 
        ('clf', DecisionTreeClassifier(random_state=42, max_depth=4))]),
    'Random Forest': Pipeline([('prep', preprocessor), 
        ('clf', RandomForestClassifier(random_state=42, n_estimators=100, max_depth=5))]),
    'Gradient Boosting': Pipeline([('prep', preprocessor), 
        ('clf', GradientBoostingClassifier(random_state=42, n_estimators=100, max_depth=3, learning_rate=0.1))]),
}

results = {}
for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    y_prob = pipe.predict_proba(X_test)[:, 1]
    
    cv_scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring='roc_auc')
    test_auc = roc_auc_score(y_test, y_prob)
    
    results[name] = {
        'test_auc': test_auc,
        'cv_auc_mean': cv_scores.mean(),
        'cv_auc_std': cv_scores.std(),
        'y_pred': y_pred,
        'y_prob': y_prob
    }
    print(f"\n{name}")
    print(f"  CV AUC: {cv_scores.mean():.3f} (+/- {cv_scores.std():.3f})")
    print(f"  Test AUC: {test_auc:.3f}")
    print(classification_report(y_test, y_pred, target_names=['Retained', 'Lapsed']))

In [ ]:
# Hyperparameter tuning for best model (Gradient Boosting)
param_grid = {
    'clf__n_estimators': [50, 100, 200],
    'clf__max_depth': [2, 3, 4],
    'clf__learning_rate': [0.05, 0.1, 0.2]
}

gb_pipe = Pipeline([('prep', preprocessor), 
    ('clf', GradientBoostingClassifier(random_state=42))])

grid_search = GridSearchCV(gb_pipe, param_grid, cv=5, scoring='roc_auc', n_jobs=-1)
grid_search.fit(X_train, y_train)

print(f"Best parameters: {grid_search.best_params_}")
print(f"Best CV AUC: {grid_search.best_score_:.3f}")

best_model = grid_search.best_estimator_
y_pred_best = best_model.predict(X_test)
y_prob_best = best_model.predict_proba(X_test)[:, 1]
print(f"Test AUC: {roc_auc_score(y_test, y_prob_best):.3f}")

In [ ]:
# Feature importance from best model
feat_imp = pd.DataFrame({
    'feature': feature_names,
    'importance': best_model.named_steps['clf'].feature_importances_
}).sort_values('importance', ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=feat_imp.head(12), x='importance', y='feature', ax=ax, palette='viridis')
ax.set_title('Feature Importance (Gradient Boosting)')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

## 4. Evaluation & Interpretation

### Metrics
We evaluate using AUC-ROC (handles class imbalance), precision, recall, and F1. We also examine the confusion matrix to understand the real-world consequences of errors.

### Business Interpretation of Errors
- **False Positive** (model says "at risk" but donor would have stayed): Staff sends an unnecessary thank-you or outreach message. **Cost: minimal** — the donor receives extra appreciation, which may even strengthen the relationship.
- **False Negative** (model says "safe" but donor actually lapses): The organization loses a donor without any intervention attempt. **Cost: high** — lost revenue, potentially permanently, with no chance to re-engage.

Given this asymmetry, we prefer a model with **high recall** (catch most at-risk donors) even at the expense of some precision (accepting a few false alarms).

In [ ]:
# ROC curves for all models
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for name, r in results.items():
    fpr, tpr, _ = roc_curve(y_test, r['y_prob'])
    axes[0].plot(fpr, tpr, label=f"{name} (AUC={r['test_auc']:.3f})")
axes[0].plot([0,1],[0,1], 'k--', alpha=0.5)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curves')
axes[0].legend()

# Confusion matrix for best model
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_best, 
    display_labels=['Retained', 'Lapsed'], cmap='Blues', ax=axes[1])
axes[1].set_title('Confusion Matrix (Best Model)')

plt.tight_layout()
plt.show()

In [ ]:
# Model comparison summary
comparison = pd.DataFrame({
    'Model': list(results.keys()),
    'CV AUC Mean': [r['cv_auc_mean'] for r in results.values()],
    'CV AUC Std': [r['cv_auc_std'] for r in results.values()],
    'Test AUC': [r['test_auc'] for r in results.values()]
}).sort_values('Test AUC', ascending=False)

print("Model Comparison Summary")
print("=" * 60)
print(comparison.to_string(index=False))

## 5. Causal and Relationship Analysis

### Key Findings from the Explanatory Model

The logistic regression coefficients reveal the following relationships:

1. **Recency is the strongest signal**: Donors who have not given recently are dramatically more likely to lapse. This is both intuitive and actionable — a simple recency threshold can serve as a first-pass alert.

2. **Recurring giving is protective**: Donors with recurring commitments are significantly less likely to lapse. This suggests the organization should make it easy and attractive to set up recurring donations.

3. **Frequency and total giving matter**: Donors who have given multiple times and at higher total amounts are more engaged and less likely to leave. This may reflect deeper emotional connection to the mission.

4. **Campaign participation**: Donors who have participated in named campaigns show different retention patterns. Campaigns may serve as engagement touchpoints that keep donors connected.

5. **Acquisition channel effects**: The channel through which a donor was acquired may influence long-term retention. Some channels bring "stickier" donors than others, which has implications for where the organization invests its limited outreach time.

### Causal Defensibility
We must be honest about what we can and cannot claim:

- **Correlation, not causation**: We observe that recurring donors lapse less, but we cannot prove that *making* a donation recurring *causes* retention. It may be that donors who are already more committed choose recurring — a selection effect.
- **Recency as a leading indicator**: High recency (long time since last gift) predicts lapse, but it may partly be *definitional* — someone who hasn't given in a long time is closer to the lapse threshold by construction. The operational value is still real, but the causal mechanism is circular.
- **Acquisition channel**: Any observed differences could reflect the type of person each channel attracts, not a direct effect of the channel itself.

### Feature Importance from the Predictive Model
The gradient boosting feature importances largely confirm the explanatory model's findings but may weight features differently due to nonlinear interactions. Both models agree that recency, frequency, and recurring status are the most informative features.

### Recommendations Based on Findings
1. **Implement a recency-based alert**: Flag donors who have not given in 90+ days for outreach.
2. **Promote recurring giving**: Make it a standard ask in every campaign and thank-you communication.
3. **Track campaign participation**: Donors who engage with campaigns appear more retained; use campaigns as touchpoints.
4. **Monitor acquisition channels**: Track which channels produce the most durable donors and invest accordingly.

## 6. Deployment Notes

### How This Model Is Deployed
The trained model is serialized using `joblib` and served through a .NET API endpoint. When the Donors & Contributions page loads, the backend:
1. Computes features for each active donor from the database.
2. Passes features through the trained model.
3. Returns a lapse risk score (probability) and category (High/Medium/Low) for each donor.

### Web App Integration
- **Donors & Contributions page**: A "Lapse Risk" column appears next to each active donor with a color-coded badge (red = High, yellow = Medium, green = Low). Staff can sort and filter by risk level to prioritize outreach.
- **Admin Dashboard**: A KPI card shows "X donors at high risk of lapsing" as an at-a-glance metric.
- **Threshold**: High risk >= 0.7 probability, Medium = 0.4-0.7, Low < 0.4. These thresholds can be adjusted by the organization.

### Model Export

In [ ]:
# Export the best model for deployment
joblib.dump(best_model, 'donor_churn_model.pkl')
print("Model saved to donor_churn_model.pkl")

# Also save the feature configuration for the API
model_config = {
    'numeric_features': numeric_features,
    'categorical_features': categorical_features,
    'reference_date': str(REFERENCE_DATE.date()),
    'lapse_window_days': LAPSE_WINDOW_DAYS,
    'risk_thresholds': {'high': 0.7, 'medium': 0.4, 'low': 0.0}
}
import json
with open('donor_churn_config.json', 'w') as f:
    json.dump(model_config, f, indent=2)
print("Config saved to donor_churn_config.json")

In [ ]:
# Example: predict risk for all current donors
all_donors = donor_features[numeric_features + categorical_features].copy()
donor_features['lapse_probability'] = best_model.predict_proba(all_donors)[:, 1]
donor_features['risk_category'] = pd.cut(
    donor_features['lapse_probability'], 
    bins=[0, 0.4, 0.7, 1.0], 
    labels=['Low', 'Medium', 'High']
)

print("Risk Distribution:")
print(donor_features['risk_category'].value_counts())
print()
print("Sample predictions:")
print(donor_features[['supporter_id', 'recency_days', 'donation_count', 'total_amount', 
                       'has_recurring', 'lapse_probability', 'risk_category']].head(10).to_string(index=False))